## **Carregar bibliotecas**

In [1]:
# Load the used classes
from modules_otft._imports import *
from modules_otft._read_data import ReadData
from modules_otft._model import TFTModel
from modules_otft._grafics import  TFTGraphicsPlot
from modules_otft._menu import TFTMenu

# from modules_otft._global_vars import *
from modules_otft._config import *
from modules_otft._utils import *
from modules_otft._display import display_settings

## **Gerar dados sintéticos**   

In [2]:
from modules_otft._generate_synthetic_data import SyntheticFromSettings

plot = TFTGraphicsPlot()
menu = TFTMenu()
read = ReadData()



In [3]:
# Read and show the path file Json
settings = enter_with_json_file()

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------


___________________________________________________________________________________ SETTINGS PRESENT IN THE JSON FILE:___________________________________________________________________________________

| path: /home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/datas/Org3/tipo p
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
| experimental_data_scale_transfer: A
---------------------------------------------------------------------------

In [4]:
ld_voltages = get_load_voltages(settings)

type_curve_plot = get_type_plot(settings)

shift_list = calculate_shift_list(settings)

list_tension_shift = get_shift_list(read, settings)

path_voltages = read.read_files_experimental(settings['path'], list_tension_shift)

Vv, Id, input_voltage, n_points, count_transfer, count_output = read.load_data(settings['type_read_data_exp'], path_voltages, settings['current_typic'], settings['experimental_data_scale_transfer'], settings['experimental_data_scale_output'], type_curve_plot)


In [5]:
# Caso o Shift seja aplicado 
path_voltages, new_values_tension, new_list_tension = filter_and_load_files(read, settings, path_voltages, list_tension_shift)

Vv, Id, input_voltage, n_points, count_transfer, count_output = read.load_data(settings['type_read_data_exp'], path_voltages, settings['current_typic'], settings['experimental_data_scale_transfer'], settings['experimental_data_scale_output'], type_curve_plot)
list_tension_shift = new_values_tension
list_tension = new_list_tension


In [6]:
Model = TFTModel(list_tension, n_points)


In [7]:
gen = SyntheticFromSettings(settings, model_cls=TFTModel, out_root="synthetic_from_settings")

In [10]:
# gerar 8 variações sintéticas por cada arquivo experimental encontrado
generated_index = gen.generate(n_variations_per_file=0, resample_n_points=10000, variation_mode="voltages", voltage_increment=2.0, voltage_max_variation=70, save_csv=True, save_npz=True)


Gerado: synthetic_from_settings/Org3/tipo_p/output/output-neg18V_synth_000.csv | Tensão: -18.0 V | params(Vtho=5.53, Jth=0.4)
Gerado: synthetic_from_settings/Org3/tipo_p/output/output-neg20V_synth_001.csv | Tensão: -20.0 V | params(Vtho=5.53, Jth=0.4)
Gerado: synthetic_from_settings/Org3/tipo_p/output/output-neg22V_synth_002.csv | Tensão: -22.0 V | params(Vtho=5.53, Jth=0.4)
Gerado: synthetic_from_settings/Org3/tipo_p/output/output-neg24V_synth_003.csv | Tensão: -24.0 V | params(Vtho=5.53, Jth=0.4)
Gerado: synthetic_from_settings/Org3/tipo_p/output/output-neg26V_synth_004.csv | Tensão: -26.0 V | params(Vtho=5.53, Jth=0.4)
Gerado: synthetic_from_settings/Org3/tipo_p/output/output-neg28V_synth_005.csv | Tensão: -28.0 V | params(Vtho=5.53, Jth=0.4)
Gerado: synthetic_from_settings/Org3/tipo_p/output/output-neg30V_synth_006.csv | Tensão: -30.0 V | params(Vtho=5.53, Jth=0.4)
Gerado: synthetic_from_settings/Org3/tipo_p/output/output-neg32V_synth_007.csv | Tensão: -32.0 V | params(Vtho=5.53, J

## **Treinar modelo MLP**

In [11]:
from modules_otft._mlp_train import MLPModelTrain 

In [12]:

BASE_PATH = "/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/"
INDEX_PATH = BASE_PATH + "synthetic_from_settings/Org3/tipo_p/index_generated_exp1_org3.json"
TECNOLOGY = "org3"

In [ ]:
# Definição dos parâmetros do treino
HIDDEN_LAYERS = (3, 3)
EPOCS = 5000  # Máximo de épocas
TEST_SPLIT = 0.2      # 20% para teste/validação
LEARNING_RATE = 0.0001
BATSIZE = 128
PATIENCE = 200


# Instanciação do Otimizador/Treinador
mlp_trainer = MLPModelTrain(
    index_path=INDEX_PATH,
    hidden_layers=HIDDEN_LAYERS,
    activation='tanh',
    learning_rate=LEARNING_RATE,
    max_iter=EPOCS,
    batch_size=BATSIZE,
    n_iter=PATIENCE,
    test_size=TEST_SPLIT,
    model_path=f'models/mlp_model_{TECNOLOGY}.pkl',
    
)

print(f"Estrutura da MLP definida: Input (5 features) -> {HIDDEN_LAYERS[0]} -> {HIDDEN_LAYERS[1]} -> Output (ln|Id|)")

Estrutura da MLP definida: Input (5 features) -> 3 -> 3 -> Output (ln|Id|)


In [14]:
# Inicia o processo de treinamento
# Este método retorna o modelo treinado e o scaler_X ajustado
mlp_trainer.train_model()

Carregando índice de dados em: /home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/synthetic_from_settings/Org3/tipo_p/index_generated_exp1_org3.json
Padronizando Features (X) e Target (Y)...
Iniciando treinamento customizado da MLP (Scikit-Learn)...
Arquitetura: Input(5) -> (3, 3, 3) -> Output(1)
Total de Parâmetros Treináveis: 46
Epoch 1/5000 - loss: 1.1591 - val_loss: 1.1866 - acc: -0.1648 - val_acc: -0.1639
Epoch 2/5000 - loss: 1.0602 - val_loss: 1.0853 - acc: -0.0654 - val_acc: -0.0646
Epoch 3/5000 - loss: 0.9450 - val_loss: 0.9672 - acc: 0.0504 - val_acc: 0.0512
Epoch 4/5000 - loss: 0.8018 - val_loss: 0.8208 - acc: 0.1943 - val_acc: 0.1949
Epoch 5/5000 - loss: 0.6786 - val_loss: 0.6946 - acc: 0.3181 - val_acc: 0.3186
Epoch 6/5000 - loss: 0.5908 - val_loss: 0.6044 - acc: 0.4063 - val_acc: 0.4071
Epoch 7/5000 - loss: 0.5285 - val_loss: 0.5401 - acc: 0.4689 - val_acc: 0.4702
Epoch 8/5000 - loss: 0.4823 - val_loss: 0.4923 - acc: 0.5153 - val_acc: 0.5171
Epoch 9/5000 - loss: 0.4453 -

In [15]:
# Carregar o modelo treinado para inferência
trained_model, feature_scaler, target_scaler = mlp_trainer.load_model(filename=f'models/Mlp_Org3/exp1/mlp_model_{TECNOLOGY}.pkl')

In [20]:
# Importações necessárias 
from modules_otft._inferency import plot_curve_comparison, prepare_mlp_features, load_data_for_inference
import numpy as np


In [21]:
# Arquivo CSV de teste e seus metadados
TEST_CSV_PATH = '/home/rsb6/Desktop/TCC/TCC implementation/Model_OTFT/models/infer_curve.json'
V_real, I_real, v_fixed, is_transfer = load_data_for_inference(TEST_CSV_PATH)



In [22]:

plot_curve_comparison(
    V_data=V_real,
    I_real=I_real,  
    V_fixed=v_fixed,
    is_transfer=is_transfer,
    
    # Passando os objetos treinados: (Função de preparo, Modelo, Scaler)
    mlp_model=(prepare_mlp_features, trained_model, feature_scaler, target_scaler), 
    
    yscale="log",  # Escala linear para corrente
    title=f"Previsão MLP vs. Curva Real ({os.path.basename(TEST_CSV_PATH)})"
)

In [30]:

plot_curve_comparison(
    V_data=V_real,
    I_real=I_real,  
    V_fixed=v_fixed,
    is_transfer=is_transfer,
    
    # Passando os objetos treinados: (Função de preparo, Modelo, Scaler)
    mlp_model=(prepare_mlp_features, trained_model, feature_scaler, target_scaler), 
    
    yscale="log",  # Escala linear para corrente
    title=f"Previsão MLP vs. Curva Real ({os.path.basename(TEST_CSV_PATH)})"
)